# Streamlit Demo

This notebook checks and launches the local sign-in page using the saved models.

Take a photo and let the camera model identify the enrolled person. Upload that person's voice recording and then their fingerprint image from `datasets/demo`. The prepared samples represent virtual identities.

In [ ]:
import sys
import subprocess
import pandas as pd
from IPython.display import display, Code
from config import ROOT
from verification import load_system

assert (ROOT / 'app.py').exists(), 'The shared Streamlit application is missing.'
system = load_system()
identities = pd.DataFrame.from_dict(system['identities'], orient='index')
display(identities[['display_name', 'voice_identity', 'fingerprint_identity']])
print('Saved models:', system['selected'])
print('Python:', sys.executable)

## The Actual Application Code

The code below is read directly from `app.py`. The camera supplies the face image. The camera face model locates and crops the face, normalizes contrast, and extracts HOG/LBP. Training uses geometric and lighting variations of the original training portraits. Standardization and PCA fit the training features. Validation selects between KNN and linear SVM and sets the acceptance threshold.

A face match within the saved threshold identifies the person. Each successful check advances to the next step, and the final step confirms sign-in.


In [ ]:
app_source = (ROOT / 'app.py').read_text(encoding='utf-8')
compile(app_source, str(ROOT / 'app.py'), 'exec')
display(Code(app_source, language='python'))

## Check the Sign-in Steps with Demo Samples

The Streamlit test runner supplies the reference photo at the camera widget and uploads the two demo files. The page uses the saved models for each check. This exercises the sign-in logic with prepared samples.


In [ ]:
from streamlit.testing.v1 import AppTest

observed = []
for identity in system['identities'].values():
    name = identity['display_name']
    folder = ROOT / 'datasets' / 'demo' / name
    page = AppTest.from_file(str(ROOT / 'app.py'), default_timeout=60).run()
    assert page.camera_input and not page.selectbox and not page.file_uploader

    page.camera_input[0].upload('face.jpg', (folder / 'face.jpg').read_bytes()).run()
    assert [widget.label for widget in page.file_uploader] == ['Voice recording']
    assert f'Identified as {name.title()}.' in [message.value for message in page.success]
    page.file_uploader[0].set_value(('voice.wav', (folder / 'voice.wav').read_bytes(), 'audio/wav')).run()

    assert [widget.label for widget in page.file_uploader] == ['Fingerprint image']
    page.file_uploader[0].set_value(('fingerprint.png', (folder / 'fingerprint.png').read_bytes(), 'image/png')).run()
    assert not page.exception and not page.error
    assert page.success[-1].value == f'Signed in as {name.title()}.'
    observed.append({'account': name.title(), 'result': page.success[-1].value})

display(pd.DataFrame(observed))

## Start the Local Server
**Input:** the selected project interpreter. **Operation:** launch Streamlit in a separate process if START_SERVER is set to True. **Output:** a local URL. If port 8501 is occupied, choose another PORT.

The default is False so Run All does not create duplicate servers. A server may already have been started during project verification. You can also use the exact printed command in a terminal. Stop only the process started by this notebook with the final cell.

In [ ]:
PORT = 8501
START_SERVER = False
command = [sys.executable, '-m', 'streamlit', 'run', str(ROOT / 'app.py'),
           '--server.port', str(PORT), '--server.address', '127.0.0.1']
print(subprocess.list2cmdline(command))
print(f'Local URL: http://127.0.0.1:{PORT}')
if START_SERVER:
    if 'server_process' in globals() and server_process.poll() is None:
        print('The server started by this notebook is already running.')
    else:
        server_process = subprocess.Popen(command, cwd=ROOT)
        print('Started server process:', server_process.pid)

## Stop This Notebook's Server

Leave `STOP_SERVER = False` to keep the demo available. The stop cell applies to the process started by this notebook.


In [ ]:
STOP_SERVER = False
if STOP_SERVER and 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=15)
    print('Stopped this notebook server.')
else:
    print('No server stopped.')